In [ ]:
include("../RayTracing.jl")

In [ ]:
LA = RayTracing.LookAt(
    RayTracing.Pnt3(0, 2, 5),
    RayTracing.Pnt3(.02, .47, 0),
    RayTracing.Vec3(0, 1, 0),
)
curves_t = RayTracing.Inv(RayTracing.Translate(RayTracing.Pnt3(0,2,5))) * RayTracing.Translate(RayTracing.Pnt3(.15, -.03, 0)) * RayTracing.Scale(7.0, 7.0, 7.0)
fur_shape_core = RayTracing.ShapeCore(
    curves_t,
    RayTracing.Inv(curves_t),
    false,
    false
)

# curve = RayTracing.parse_curves(
#     RayTracing.jmfp("/home/jmyslinski/random_stuff/pbrt-v4-scenes/bunny-fur/geometry/bunny-fur-curves-small.pbrt"),
#     identity_shape_core
# )[1]
curve = RayTracing.CreateCurve(
    fur_shape_core,
    RayTracing.SVector(
        RayTracing.Pnt3( -0.048298, 0.076975, -0.018403 ),
        RayTracing.Pnt3(-0.047604, 0.078775, -0.022468 ),
        RayTracing.Pnt3(-0.04686, 0.077952, -0.026829),
        RayTracing.Pnt3( -0.046247, 0.075309, -0.030419 ),
    ),
    8e-05,
    7e-06,
    "flat",
    nothing,
    3
)[7]

In [ ]:
# equals worldfromcamera
LA

In [ ]:
hacky_t = RayTracing.Inv(RayTracing.Translate(RayTracing.Pnt3(0,2,5))) * RayTracing.Translate(RayTracing.Pnt3(.15, -.03, 0)) * RayTracing.Scale(7.0, 7.0, 7.0)

In [ ]:
curve.u_min

In [ ]:
curve.u_max

In [ ]:
u = 0.1
p = RayTracing.blossom_cubic_bezier(curve.common.cp_obj, u, u, u)

In [ ]:
o = RayTracing.Pnt3(2.451949, -1.669419, -11.742881)
d = RayTracing.Vec3( -0.37262046, 0.024686506, 0.92765534)
r = RayTracing.Ray(
    o,
    d,
    0.0,
    typemax(Float64)
)

# begin matching

In [ ]:
ray = RayTracing.Inv(hacky_t)(r)

In [ ]:
cp_obj = RayTracing.cubic_bezier_control_points(curve.common.cp_obj, curve.u_min, curve.u_max)

In [11]:
dx = RayTracing.cross(ray.direction, cp_obj[3+1]- cp_obj[0+1])
if RayTracing.length_squared(dx) == 0.0
    _, dx, dy = RayTracing.orthonormal_basis(ray.direction)
end

In [ ]:
ray_from_object = RayTracing.Inv(RayTracing.LookAt(ray.origin, ray.origin + ray.direction, dx))

In [ ]:
cp = RayTracing.SVector(
    ray_from_object(cp_obj[0+1]),
    ray_from_object(cp_obj[1+1]),
    ray_from_object(cp_obj[2+1]),
    ray_from_object(cp_obj[3+1]),
)

In [ ]:
max_width = max(
    RayTracing.lerp(curve.u_min, curve.common.width.x, curve.common.width.y),
    RayTracing.lerp(curve.u_max, curve.common.width.x, curve.common.width.y),
)
curve_bounds = RayTracing.expand(RayTracing.world_bounds(RayTracing.Bounds3(cp[0+1], cp[1+1]), RayTracing.Bounds3(cp[2+1], cp[3+1])), 0.5 * max_width)

In [ ]:
ray_bounds = RayTracing.Bounds3(
    RayTracing.Pnt3(0, 0, 0),
    RayTracing.Pnt3(0, 0, RayTracing.length_pbrt(ray.direction) * ray.tMax)
)

In [ ]:
RayTracing.overlaps(ray_bounds, curve_bounds)

# real call

In [ ]:
RayTracing.intersect(curve, r)